In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader
import os
import gradio as gr
from pydantic import BaseModel
import requests
import json

In [ ]:
class Evaluation(BaseModel):
    is_acceptable : bool
    feedback : str


In [ ]:
load_dotenv(override=True)
client = OpenAI(
    base_url= "https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

In [ ]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
webhook_url = os.getenv("WEBHOOK_URL")
print(webhook_url)

In [ ]:
def send_discord_message(message):
    data = {"content": message}
     
    response = requests.post(webhook_url, json=data)
    if response.status_code == 204:
        print("Alert sent to Discord!")
    else:
        print(f"Failed: {response.status_code}, {response.text}")


In [ ]:
send_discord_message("Hello, Faisal!, I am still in testing mode")

In [ ]:
def record_user_details(email, name= "Name not provided", notes= "Notes not provided"):
    send_discord_message(f"New user details recorded:\nEmail: {email}\nName: {name}\nNotes: {notes}")
    return (f"Details for {name} recorded successfully!")

In [ ]:
def record_unknown_question(question):
    send_discord_message(f"New unknown question recorded: {question}")
    return (f"Question '{question}' recorded successfully!")


In [ ]:
record_user_details_json = {
    "name" : "record_user_details",
    "description" : "use this tool to record that a user is interested being in touch with me and provided an email to connect to me",
    "parameters":{
        "type": "object",
        "properties":{
            "email":{
                "type": "string",
                "description": "The email address of this user"
            },
            "name":{
                "type": "string",
                "description": "The name of the user, if the user provided it"
            },
            "notes":{
                "type": "string",
                "description": "Any additional notes about the user, such as their company or position or any other information about the conversation that is worth recording for my later user and for my database"
            }
        
        },
        "required": ["email"],
        "additionalProperties": False
    }
}







In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "use this tool to record that a user asked a question that is not related to my website or my professional background",
    "parameters":{
        "type": "object",
        "properties":{
            "question":{
                "type": "string",
                "description": "The question that the user asked"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}   

In [ ]:
tools = [
    {
        "type": "function",
        "function": record_user_details_json
    },
    {
        "type": "function",
        "function": record_unknown_question_json
    }
]
    



In [ ]:
tools

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Executing tool: {tool_name}", flush=True)
        tool = globals()[tool_name]
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})  
    return results

In [ ]:
reader = PdfReader("about-me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

print(linkedin)

In [ ]:
with open("about-me/my-profile-summary.txt", "r" , encoding = "UTF-8") as f:
    summary = f.read()

print(summary)



In [ ]:
name = "Faisal Haroon"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

tool_protocols = """
# TOOL CALLING RULES
1. **NEVER** talk and call a tool in the same turn. If calling a tool, the response must be ONLY the tool call.
2. **NO HALLUCINATIONS:** If an answer isn't in the provided ## Summary or ## LinkedIn, you MUST use `record_unknown_question`.
3. **LEAD CAPTURE (record_user_details):**
   - TRIGGER: User shows project interest or prepares to leave (bye/thanks).
   - STEP 1: Ask for Name and Email in the same response and record them.
   - STEP 2: If Name and Email are provided, call the tool to record details otherwise do not call the record_user_details tool.
   - STEP 3: Call tool ONLY when both are provided.
   - STEP 4: This is the main rule only ask for thr user details when he is interested for a project or any question that is unknown and not in the provided context or either the user is leaving by saying the leaving words (for example Thanks , Thank you, Bye, nice talking to you).
   - FORBIDDEN: Do not use placeholders like "null" or "user@example.com please make sure to call the tool only when the user provides both. ".
"""

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."
system_prompt += tool_protocols

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role":"user", "content":"Do you hold a patent?"}]
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",   # upgrade from 8b — more reliable tool calling
    messages=messages
)
reply = response.choices[0].message.content
print(reply)

In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# def evaluate(reply, message, history) -> Evaluation:

#     messages= [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
#     response = gemini.chat.completions.parse(
#         model="gemini-3-flash-preview",
#         messages=messages,
#         response_format=Evaluation
#     )
#     evaluation_answer = response.choices[0].message.parsed
#     return evaluation_answer

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
     history = [{"role": h["role"], "content": h["content"]} for h in history]
     messages= [{"role": "system", "content": system_prompt}] + history[-6:] + [{"role": "user", "content": message}]
     done = False
     while not done:
     
      response = client.chat.completions.create(
         model="llama-3.1-8b-instant",
         messages=messages,
         tools=tools
         )
      finish_reason = response.choices[0].finish_reason
 
      if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
      else:
            done = True
            return response.choices[0].message.content


In [ ]:
gr.ChatInterface(chat).launch()